In [1]:
import cobra
import escher
import pandas as pd
import numpy as np
import json
import csv
from cobra.io import save_json_model
from cobra import Reaction
from cobra.flux_analysis.loopless import add_loopless, loopless_solution

In [2]:
# model_path = 'iMT1026v3.xml'
model_path ='..\model\iMT1026v3jup.xml'
model = cobra.io.read_sbml_model(model_path)

model

Set parameter Username
Academic license - for non-commercial use only - expires 2026-10-13


Name,iMT1026v3
Memory address,1c09d6f3050
Number of metabolites,1706
Number of reactions,2237
Number of genes,1026
Number of groups,77
Objective expression,1.0*Ex_biomass - 1.0*Ex_biomass_reverse_5354f
Compartments,"Vacuole, Cytosol, Mitochondria, Peroxisome, Extracellular space, Endoplasmic Reticulum, Golgi Apparatus, Nucleus, Mitochondrial intermembrane space"


In [3]:
# Change grwoth on glycerol to growth on 60% glucose and 40% methanol
# Add reaction describing this growth

biomass_gly = model.reactions.get_by_id('BIOMASS_glyc')
biomass_gly.bounds = (0,0)
biomass_gly

carbs = model.metabolites.get_by_id('CARBOHYDRATES_c')
DNA = model.metabolites.get_by_id('DNA_c')
lipids = model.metabolites.get_by_id('LIPIDS_c')
prot = model.metabolites.get_by_id('PROTEIN_c')
RNA = model.metabolites.get_by_id('RNA_c')
atp = model.metabolites.get_by_id('atp_c')
cof = model.metabolites.get_by_id('cof_c')
h2o = model.metabolites.get_by_id('h2o_c')
adp = model.metabolites.get_by_id('adp_c')
biomass = model.metabolites.get_by_id('biomass_c')
h = model.metabolites.get_by_id('h_c')
pi = model.metabolites.get_by_id('pi_c')

biomass_glc_meoh = Reaction ('biomass_glc_meoh_60/40')
biomass_glc_meoh.name = 'Biomass composition (g/g) - 60/40 Glucose/Methanol'
biomass_glc_meoh.add_metabolites({carbs: -0.33,
                                   DNA: -0.001,
                                   lipids: -0.042,
                                   prot: -0.49,
                                   RNA: -0.058,
                                   atp: -63.85,
                                   cof: -1,
                                   h2o: -63.85,
                                   adp: 63.85,
                                   pi: 63.85,
                                   biomass: 1,
                                   h: 63.85})
biomass_glc_meoh

Reaction identifier,biomass_glc_meoh_60/40
Name,Biomass composition (g/g) - 60/40 Glucose/Methanol
Memory address,0x1c0a5b77dd0
Stoichiometry,0.33 CARBOHYDRATES_c + 0.001 DNA_c + 0.042 LIPIDS_c + 0.49 PROTEIN_c + 0.058 RNA_c + 63.85 atp_c + cof_c + 63.85 h2o_c --> 63.85 adp_c + biomass_c + 63.85 h_c + 63.85 pi_c 0.33 Carbohydrates + 0.001 DNA + 0.042 Lipids + 0.49 PROTEIN + 0.058 RNA + 63.85 ATP + Cofactors and small molecules + 63.85 H2O --> 63.85 ADP + Biomass + 63.85 H+ + 63.85 Phosphate
GPR,
Lower bound,0.0
Upper bound,1000.0


In [4]:
# Add new biomass reaction to the model
print (len(model.reactions))
model.add_reactions([biomass_glc_meoh])
print (len(model.reactions))

2237
2238


In [5]:
glycerol_exchange = model.exchanges.get_by_id('Ex_glyc')
glycerol_exchange.bounds = (0,0)
glycerol_exchange

Reaction identifier,Ex_glyc
Name,Glycerol exchange
Memory address,0x1c0a64520d0
Stoichiometry,glyc_e --> Glycerol -->
GPR,
Lower bound,0
Upper bound,0


In [6]:
# Add qmeoh constraints observed in chemostat cultivations
methanol_exchange = model.exchanges.get_by_id('Ex_meoh')
methanol_exchange.bounds = (-1.24, -1.18)
methanol_exchange

Reaction identifier,Ex_meoh
Name,Methanol exchange
Memory address,0x1c0a6394250
Stoichiometry,meoh_e <-- Methanol <--
GPR,
Lower bound,-1.24
Upper bound,-1.18


In [7]:
# Add qgluc constraints observed in chemostat cultivations
glucose_exchange = model.reactions.get_by_id('Ex_glc_D')
glucose_exchange.bounds = (-0.72, -0.7)
glucose_exchange

Reaction identifier,Ex_glc_D
Name,D-Glucose exchange
Memory address,0x1c0a64e3490
Stoichiometry,glc_D_e <-- D-Glucose <--
GPR,
Lower bound,-0.72
Upper bound,-0.7


In [8]:
# Change the reactions for the synthesis of lipids, proteins and sterols from glycerol to those from glucose

model.reactions.get_by_id('LIPIDS_glyc').bounds = (0,0)
model.reactions.get_by_id('PROTEINS_glyc').bounds = (0,0)
model.reactions.get_by_id('STEROLS_glyc').bounds = (0,0)

# note: these reactions from glucose do not have the glucose specification
model.reactions.get_by_id('LIPIDS').bounds = (0,1000)
model.reactions.get_by_id('PROTEINS').bounds = (0,1000)
model.reactions.get_by_id('STEROLS').bounds = (0,1000)

In [9]:
# ATP maintenance requirement (NGAME = Non-Growth Associated Maintenance Energy) based on previous studies on 
# the growth of X33-ROL on Gluc/MeOH from the group (Eric's Master Thesis)

model.reactions.get_by_id('ATPM').bounds = (1.96, 1000)

In [10]:
rolAA = model.reactions.get_by_id('rolAA')
rolAA.bounds = (0,1000)
rolRNA = model.reactions.get_by_id('rolRNA')
rolRNA.bounds = (0,1000)
rolDNA = model.reactions.get_by_id('rolDNA')
rolDNA.bounds = (0,1000)
pROL = model.reactions.get_by_id('pROL')
pROL.bounds = (0,1000)
Rol_transport =  model.reactions.get_by_id('ROLt')
Rol_transport.bounds = (0,1000)
ROL_exchange = model.exchanges.get_by_id('Ex_rol')
ROL_exchange.bounds = (0.003,1000) 


pFAB = model.reactions.get_by_id('pFAB')
pFAB.bounds = (0,0)
fabAA = model.reactions.get_by_id('fabAA')
fabAA.bounds = (0,0)
fabt = model.reactions.get_by_id('fabt')
fabt.bounds = (0,0)
fabRNA = model.reactions.get_by_id('fabRNA')
fabRNA.bounds = (0,0)
fabDNA = model.reactions.get_by_id('fabDNA')
FAB_exchange = model.exchanges.get_by_id('Ex_fab')
FAB_exchange.bounds = (0,0)


# Extra reactions that must be closed for simulations to run smoothly:

APAT2r = model.reactions.get_by_id('APAT2r')
APAT2r.bounds = (0,0) # reaction not present in Pichia, it is yet to be removed

MMSAD3 = model.reactions.get_by_id('MMSAD3')
MMSAD3.bounds = (0,0) # The reduction reaction of MSA into Acetil-CoA it is due to an unspecific effect. Reaction
# under evaluation of being kept or not.

In [11]:
#Creating the AtOX reaction
q6h2_m = model.metabolites.get_by_id('q6h2_m')
o2_m = model.metabolites.get_by_id('o2_m')
q6_m = model.metabolites.get_by_id('q6_m')
h2o_m = model.metabolites.get_by_id('h2o_m')

AtOX = Reaction('AtOX')
AtOX.name = 'Alternative oxidase from Histoplasma capsulatum'
AtOX.add_metabolites({q6h2_m: -2,
                       o2_m: -1,
                       q6_m: 2,
                       h2o_m: 2})
#AtOX.bounds = (1.5,1.75) #Aquests son els de la Natalia
AtOX.bounds = (0,0)
print(AtOX.reaction)

# UniProt ID: Q9Y711 (Johnson et al., 2003)
# Possible Brenda ID for reaction stoichiometry: EC 1.10.3.11
# It is an enzyme from the mitochondrial inner membrane
# AtOX bounds derived from experimental data (Monforte et al., 2019) and the simulations for the overexpression of AtOX

o2_m + 2 q6h2_m --> 2 h2o_m + 2 q6_m


In [12]:
print (len(model.reactions))
model.add_reactions([AtOX])
print (len(model.reactions))

2238
2239


In [13]:
#Creating the cPOS5 reaction
atp_c = model.metabolites.get_by_id('atp_c')
nadh_c = model.metabolites.get_by_id('nadh_c')
adp_c = model.metabolites.get_by_id('adp_c')
nadph_c = model.metabolites.get_by_id('nadph_c')
h_c = model.metabolites.get_by_id('h_c')

cPOS5 = Reaction('cPOS5')
cPOS5.name = 'Cytosolic NADH Kinase from Saccharomyces cerevisiae'
cPOS5.add_metabolites({atp_c: -1, 
                           nadh_c: -1, 
                           adp_c: 1, 
                           nadph_c: 1,
                             h_c: 1})
cPOS5.bounds = (0,0)
print (cPOS5.reaction)
cPOS5

atp_c + nadh_c --> adp_c + h_c + nadph_c


Reaction identifier,cPOS5
Name,Cytosolic NADH Kinase from Saccharomyces cerevisiae
Memory address,0x1c0a6493590
Stoichiometry,atp_c + nadh_c --> adp_c + h_c + nadph_c ATP + NADH --> ADP + H+ + NADPH
GPR,
Lower bound,0
Upper bound,0


In [14]:
print (len(model.reactions))
model.add_reactions([cPOS5])
print (len(model.reactions))

2239
2240


## Reaction Ratios as constraints

### The experimental data was extracted from the paper:
#### "Metabolic flux analysis of recombinant Pichia pastoris growing on different glycerol/methanol mixtures by iterative fitting of NMR-derived 13C labelling data from proteinogenic amino acids" by Joel Jordà, 2014

In [15]:
ArabitolSecretion = model.exchanges.get_by_id('Ex_abt_D')
ICL_Reaction = model.reactions.get_by_id('ICLx')
FBA_Reaction = model.reactions.get_by_id('FBA')
MAE2m_Reaction = model.reactions.get_by_id('ME2m')
MAE1m_Reaction = model.reactions.get_by_id('ME1m')
MAE1_Reaction = model.reactions.get_by_id('ME1')
MALSp_Reaction = model.reactions.get_by_id('MALSp')
CSm_Reaction = model.reactions.get_by_id('CSm')
ALCD19_Reaction = model.reactions.get_by_id('ALCD19')
MDHm_Reaction = model.reactions.get_by_id('MDHm')
PDH_Reaction = model.reactions.get_by_id('PDHcm')
NADHD_Reaction = model.reactions.get_by_id('NADH2_u6m')

In [16]:
# REACTION RATIOS PER GLUCOSA METANOL

ReactionRatio1 = model.problem.Constraint(model.reactions.CSm.flux_expression - model.reactions.ACONTm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio1)

ReactionRatio2 = model.problem.Constraint(model.reactions.AKGDam.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio2)

ReactionRatio8 = model.problem.Constraint(68*model.reactions.AKGDam.flux_expression - 46*model.reactions.CSm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio8)

ReactionRatio9 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression - model.reactions.GND.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio9)

ReactionRatio10 = model.problem.Constraint(0.8*model.reactions.MDHm.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio10)

ReactionRatio12 = model.problem.Constraint(35*model.reactions.PYK.flux_expression - 143*model.reactions.PC.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio12)

ReactionRatio13 = model.problem.Constraint(68*model.reactions.PYK.flux_expression - 143*model.reactions.PYRt2m.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio13)


ReactionRatio14 = model.problem.Constraint(40*model.reactions.GAPD.flux_expression - 145*model.reactions.FBA.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio14)


ReactionRatio16 = model.problem.Constraint(model.reactions.GAPD.flux_expression - model.reactions.PYK.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio16)

ReactionRatio20 = model.problem.Constraint(0.18*model.reactions.FALDtx.flux_expression - 0.82*model.reactions.DAS.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio20)

ReactionRatio21 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression + 0.4*model.reactions.Ex_glc_D.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio21)

In [17]:
pfba_WT = cobra.flux_analysis.pfba(model)

In [18]:
pfba_Ref_Glc60_MeOH40 = pfba_WT.fluxes

In [19]:
pfba_Ref_Glc60_MeOH40.to_excel('pfba_Ref_Glc60_MeOH40.xlsx',sheet_name = 'RefDasGip')

ModuleNotFoundError: No module named 'openpyxl'

In [ ]:
pfba_WT.fluxes['ICDHxm']

In [21]:
# Remove Reaction Ratio Constraints before starting MOMA simulations

ReactionRatioList = [ReactionRatio1, ReactionRatio2, ReactionRatio8, ReactionRatio9, ReactionRatio10,
                     ReactionRatio12, ReactionRatio13, ReactionRatio14, ReactionRatio16, ReactionRatio20, ReactionRatio21]
                    

model.remove_cons_vars(ReactionRatioList)

## 2POS5 Strain 

In [22]:
methanol_exchange.bounds = (-1.51, -1.35)
glucose_exchange.bounds = (-0.86, -0.68)
AtOX.bounds = (0, 0)
cPOS5.bounds = (0.1, 0.1)
ROL_exchange.bounds = (0.0041,0.0041)
moma_resultPOS5 = cobra.flux_analysis.moma(model,pfba_WT,0)

In [23]:
MOMA_2CPOS5 = moma_resultPOS5.fluxes

In [24]:
MOMA_2CPOS5.to_excel('MOMA_2cPOS5_Glc60_MeOH_ExpDasGip.xlsx',sheet_name = '2cPOS5')

In [25]:
MOMA_2CPOS5['ICDHxm']

0.2683332031018115

In [26]:
MOMA_2CPOS5['Ex_rol']

0.0041

## NOX Strain

In [27]:
methanol_exchange.bounds = (-0.67, -0.55)
glucose_exchange.bounds = (-1, -0.9)
ROL_exchange.bounds = (0.00188,0.00188)
cPOS5.bounds = (0, 0)
AtOX.bounds = (0.08, 0.08)
moma_resultAtOX = cobra.flux_analysis.moma(model,pfba_WT,0)

In [28]:
MOMA_NOX = moma_resultAtOX.fluxes

In [29]:
MOMA_NOX.to_excel('MOMA_HcAtOX_Glc60_MeOH_ExpDasGip.xlsx',sheet_name = 'NOX')

In [30]:
MOMA_NOX['ICDHxm']

0.2737575512793029

## POSNOX Strain

In [31]:
methanol_exchange.bounds = (-1.24, -1.24)
glucose_exchange.bounds = (-0.9, -0.88)
ROL_exchange.bounds = (0.0046,0.0046)
cPOS5.bounds = (0.1, 0.1)
AtOX.bounds = (0.08, 0.08)
moma_resultPOSNOX = cobra.flux_analysis.moma(model,pfba_WT,0)

In [32]:
MOMA_POSNOX = moma_resultPOSNOX.fluxes

In [33]:
MOMA_POSNOX.to_excel('MOMA_POSNOX_Glc60_MeOH_ExpDasGip.xlsx',sheet_name = 'POSNOX')

In [34]:
MOMA_POSNOX['ICDHxm']

0.2685722503778684